# 1 DATA PREPARATION

**Project:** Irish Company Survival: Registrations, Cohort Survival and Prediction

**Author:** Temuulen Bulgan

### Purpose of this notebook
This notebook loads the raw company register, checks its quality and turns it into one clean table of companies that the rest of the project uses for analysis, SQL and the Power BI dashboard.

### Data sources
| Dataset | Source | Content |
|---|---|---|
| Company Records (`companies.csv.zip`) | Companies Registration Office (CRO) | One row per company registered in Ireland, including dissolved companies: registration date, status, dissolution date, company type, sector code and address |

### Contents
1. Company Records: inspection
2. Company Records: cleaning
3. Saving the clean data

### Output
This notebook saves the cleaned data to `data/processed/companies_clean.csv` (one row per company registered from 2000 onwards).

## Setup

In [1]:
import pandas as pd
from pathlib import Path

# Show all columns when displaying a DataFrame
pd.set_option("display.max_columns", None)

## 1. Company Records: inspection

**Source:** Companies Registration Office (CRO), dataset "Company Records". Downloaded from opendata.cro.ie in October 2026. Licence: CC BY 4.0. Contains Irish Public Sector Data licensed under a Creative Commons Attribution 4.0 International (CC BY 4.0) licence.

**Structure according to the source:** one row per company registered in Ireland, including dissolved companies, with registration date, status, dissolution date, company type, sector codes and address.

In this section I inspect the raw file before cleaning it.

### 1.1 Inspection

#### Load the file and check its size

In [2]:
# Load the raw file
companies = pd.read_csv("../data/raw/companies.csv.zip")

# Number of rows and columns
companies.shape

(827902, 21)

The file has 827,902 rows (one per company) and 21 columns.

#### First and last rows

In [3]:
# First and last 3 rows: format of the data, and where it starts and ends
display(companies.head(3))
display(companies.tail(3))

,company_num,company_name,company_status_code,company_status,company_type_code,company_type,company_reg_date,last_ar_date,company_address_1,company_address_2,company_address_3,company_address_4,comp_dissolved_date,nard,last_accounts_date,company_status_date,nace_v2_code,eircode,company_name_eff_date,company_type_eff_date,princ_object_code
0,340217,STORM ENTERPRISES LIMITED,1151,Normal,1153,LTD - Private Company Limited by Shares,2001-03-13,2025-09-30,C/O Storm Technology,The Capel Building,Marys Abbey,",DUBLIN",NaN,2026-09-30,2024-12-31,NaN,NaN,D07 N977,2015-11-05,2016-12-01,72.20
1,340235,L.M.G. DEVELOPMENTS LIMITED,1158,Dissolved,1153,LTD - Private Company Limited by Shares,2001-03-13,2016-09-13,7 GLEBE CREST,DONEGAL TOWN,"CO. DONEGAL,DONEGAL","CO. DONEGAL, DONEGAL, IRELAND",2017-11-29,2017-09-13,2016-03-31,2017-11-29,NaN,F94H4X5,2001-03-13,2016-12-01,45.21
2,340241,M V FOODS LIMITED,1151,Normal,1153,LTD - Private Company Limited by Shares,2001-03-13,2025-12-31,UNIT 1 CLANBRASSIL CENTRE,CLANBRASSIL STREET,DUNDALK CO.LOUTH.,"DUNDALK,LOUTH",NaN,2026-12-31,2025-03-31,NaN,NaN,NaN,2001-03-13,2016-12-01,52.22


,company_num,company_name,company_status_code,company_status,company_type_code,company_type,company_reg_date,last_ar_date,company_address_1,company_address_2,company_address_3,company_address_4,comp_dissolved_date,nard,last_accounts_date,company_status_date,nace_v2_code,eircode,company_name_eff_date,company_type_eff_date,princ_object_code
827899,829192,CK CASTLE PROPERTY SERVICES LIMITED,1151,Normal,1153,LTD - Private Company Limited by Shares,2026-10-27,NaN,82 PARK STREET,NaN,NaN,"DUNDALK,LOUTH",NaN,2027-04-27,NaN,NaN,5630.0,A91EC85,2026-10-27,2026-10-27,NaN
827900,829212,ANGEL GLOBAL VOYAGER LIMITED,1151,Normal,1153,LTD - Private Company Limited by Shares,2026-10-27,NaN,THE BLACK CHURCH,ST. MARY'S PLACE,NaN,DUBLIN 7,NaN,2027-04-27,NaN,NaN,5829.0,D07P4AX,2026-10-27,2026-10-27,NaN
827901,829189,SLM DENTAL CARE LIMITED,1151,Normal,1153,LTD - Private Company Limited by Shares,2026-10-27,NaN,Birch Lane,Iskaheen,"Muff,DONEGAL",NaN,NaN,2027-04-27,NaN,NaN,8623.0,F93 RD83,2026-10-27,2026-10-27,NaN


First look: there is one row per company. The file does not start with the oldest companies: the first rows were registered on 13 March 2001 and the last rows on 27 October 2026. The company numbers are not in strict order either. The 2001 rows have a sector code in `princ_object_code` but not in `nace_v2_code`, and the 2026 rows have the opposite. `nace_v2_code` is shown with a decimal point (for example `5630.0`). Some text has extra spaces or mixed formats, for example the addresses (`,DUBLIN`, `CO. DONEGAL,DONEGAL`).

#### Column types and missing counts

In [4]:
# Column names, data types and non-missing counts
companies.info()

<class 'pandas.DataFrame'>
RangeIndex: 827902 entries, 0 to 827901
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   company_num            827902 non-null  int64  
 1   company_name           827902 non-null  str    
 2   company_status_code    827902 non-null  int64  
 3   company_status         827902 non-null  str    
 4   company_type_code      827902 non-null  int64  
 5   company_type           827902 non-null  str    
 6   company_reg_date       811710 non-null  str    
 7   last_ar_date           610066 non-null  str    
 8   company_address_1      827902 non-null  str    
 9   company_address_2      766947 non-null  str    
 10  company_address_3      572718 non-null  str    
 11  company_address_4      802369 non-null  str    
 12  comp_dissolved_date    475006 non-null  str    
 13  nard                   802906 non-null  str    
 14  last_accounts_date     501102 non-null  str    

The identifier, name, status and type columns are complete (827,902 non-null). `company_reg_date` has fewer values (811,710), so some companies have no registration date. This matters because every survival measure starts from this date. All date columns are stored as text (`str`), so they need converting to dates. `nace_v2_code` is stored as a number (`float64`), which explains the `.0` seen earlier. It should be text, because it is a code, not an amount. Both sector code columns and `comp_dissolved_date` are only partly filled. The next check counts the missing values in each column.

#### Missing values per column

In [5]:
# Number of missing values in each column
missing_count = companies.isna().sum()

# Missing values as a percentage of all rows
missing_pct = (missing_count / len(companies) * 100).round(1)

# Put both in one table, largest first
missing = pd.DataFrame({"missing": missing_count, "missing_pct": missing_pct})
missing.sort_values("missing", ascending=False)

,missing,missing_pct
nace_v2_code,561909,67.9
eircode,533605,64.5
princ_object_code,369827,44.7
comp_dissolved_date,352896,42.6
last_accounts_date,326800,39.5
company_status_date,320925,38.8
company_address_3,255184,30.8
last_ar_date,217836,26.3
company_address_2,60955,7.4
company_address_4,25533,3.1


The largest gaps are in the sector codes (`nace_v2_code` about 68% missing, `princ_object_code` about 45%) and `eircode` (about 65%). `comp_dissolved_date` is missing for about 43% of companies, which is expected if most of these are companies that are still registered. A later check confirms this against `company_status`. `company_reg_date` is missing for 16,192 companies (2.0%). The name and type effective dates have almost the same number of gaps, so these may be the same companies. Seven columns have no missing values.

#### Company status categories
This column decides which companies count as closed, so I list every value and how often it appears.

In [6]:
# Every status value and its number of companies (including missing)
companies["company_status"].value_counts(dropna=False)

company_status
Dissolved                           458801
Normal                              323502
Dissolved-20 years                   17653
Strike Off Listed                    10806
Liquidation                           9211
Ceased IRL                            2676
Dissolved PostMerger                  2100
Ceased                                1254
Normal                                 900
Struck Off                             687
Deleted CB Merger                      182
De-Registered ICAV                      68
Conv. to IndProvSoc                     22
Administration (UK)                     12
Deleted CB Conversion                   10
Deleted per SE merge                     7
Deleted as per SE Tr                     3
Examinership                             2
Liquidation (UK)                         2
Administration                           1
Liquidation stayed                       1
De-Registered Investment Company         1
Dissolved by Divis                     

#### Hidden spaces in status values
`Normal` appears twice in the table above. Here I show each value in quotes, so that spaces at the start or end are visible.m

In [7]:
# Each status value in quotes, with its number of companies
status_counts = companies["company_status"].value_counts()

for status, count in status_counts.items():
    print(repr(status), count)

'Dissolved' 458801
'Normal ' 323502
'Dissolved-20 years' 17653
'Strike Off Listed' 10806
'Liquidation' 9211
'Ceased IRL' 2676
'Dissolved PostMerger' 2100
'Ceased' 1254
'Normal' 900
'Struck Off' 687
'Deleted CB Merger' 182
'De-Registered ICAV' 68
'Conv. to IndProvSoc' 22
'Administration (UK)' 12
'Deleted CB Conversion' 10
'Deleted per SE merge' 7
'Deleted as per SE Tr' 3
'Examinership' 2
'Liquidation (UK)' 2
'Administration' 1
'Liquidation stayed' 1
'De-Registered Investment Company' 1
'Dissolved by Divis' 1


There are 23 status values. The two largest are `Dissolved` and `Normal `. `Normal` appears twice because 323,502 values have a trailing space and 900 do not. These are the same status, so cleaning will remove the spaces before grouping. Besides these, there are dissolved variants (`Dissolved-20 years`, `Dissolved PostMerger`), statuses for companies being closed (`Strike Off Listed`, `Liquidation`) and many small groups with fewer than 200 companies each. A rule is needed for which of these count as closed.

#### Company type categories
Company type is one of the features for the survival model, so I check its values.

In [8]:
# Number of different company types
print("Number of company types:", companies["company_type"].nunique())

# Every company type and its number of companies
companies["company_type"].value_counts()

Number of company types: 44


company_type
LTD - Private Company Limited by Shares                                        405121
Private limited by shares                                                      240139
Private                                                                         49224
Single member private company limited by shares                                 45474
CLG - Company Limited by Guarantee                                              22459
Unknown                                                                         15290
DAC - Designated Activity Company (limited by shares)                           11152
ULC - Private Unlimited Company                                                  7903
External company                                                                 7819
Private unlimited with share capital                                             7208
Guarantee company without a share capital (public)                               6867
Private unlimited                        

There are 44 company types. The same type appears under different labels: for example `LTD - Private Company Limited by Shares` (405,121) and `Private limited by shares` (240,139), or `PLC - Public Limited Company` and `Public limited company`. One label system uses short codes such as `LTD`, `CLG` and `DAC`, and the other uses longer descriptions. 15,290 companies have the type `Unknown`. Many types have fewer than 100 companies. For the model, the types will need grouping into a few main groups.

#### Company type by status
To see why the same type has two labels, I compare each of the 10 largest types with whether the company is still `Normal` (active).

In [9]:
# True when the status is Normal (spaces removed only for this comparison)
is_normal = companies["company_status"].str.strip() == "Normal"

# Number of companies per type, split by Normal (True) or not (False)
type_by_status = pd.crosstab(companies["company_type"], is_normal)

# Show the 10 largest types
top_types = companies["company_type"].value_counts().head(10).index
type_by_status.loc[top_types]

company_status,False,True
company_type,,
LTD - Private Company Limited by Shares,117805,287316
Private limited by shares,240091,48
Private,49223,1
Single member private company limited by shares,45459,15
CLG - Company Limited by Guarantee,4053,18406
Unknown,15290,0
DAC - Designated Activity Company (limited by shares),3891,7261
ULC - Private Unlimited Company,2622,5281
External company,4184,3635


The two label systems split by status. The long labels (`Private limited by shares`, `Private`, `Single member private company limited by shares`, `Private unlimited with share capital`) belong almost only to companies that are no longer `Normal`. The short-code labels (`LTD`, `CLG`, `DAC`, `ULC`) include most of the active companies. This suggests that the short labels are the current types and that closed companies kept their old labels. All 15,290 `Unknown` companies are not `Normal`. When cleaning, old and new labels of the same type will be mapped to one group.

#### Registration dates
The registration date is stored as text. I convert a copy to dates to check that every value is a valid date and to see the earliest and latest dates.

In [10]:
# Convert a copy of the column to dates; values that are not valid dates become missing (NaT)
reg_date = pd.to_datetime(companies["company_reg_date"], format="%Y-%m-%d", errors="coerce")

# Text values that could not be converted (filled in the raw column, missing after converting)
not_converted = companies["company_reg_date"].notna() & reg_date.isna()
print("Missing in raw file:", companies["company_reg_date"].isna().sum())
print("Could not be converted:", not_converted.sum())

# Earliest and latest registration date
print("Earliest:", reg_date.min())
print("Latest:", reg_date.max())

Missing in raw file: 16192
Could not be converted: 0
Earliest: 1753-01-01 00:00:00
Latest: 2026-10-27 00:00:00


All filled registration dates are valid dates in the same format, and 16,192 are missing. The latest date is 27 October 2026, the day before the download. The earliest date, 1 January 1753, is not a realistic registration date and looks like a placeholder. The next check looks at the oldest dates.

#### The oldest registration dates

In [11]:
# Number of companies with each of the oldest registration dates
print("Registered in 1753:", (reg_date.dt.year == 1753).sum())
print("Registered before 1900:", (reg_date < "1900-01-01").sum())

# The company registered in 1753
companies[reg_date.dt.year == 1753].T

Registered in 1753: 1
Registered before 1900: 157


,54248
company_num,960000
company_name,FORESIGHT HOLDINGS LIMITED
company_status_code,1151
company_status,Normal
company_type_code,1150
company_type,DAC - Designated Activity Company (limited by ...
company_reg_date,1753-01-01
last_ar_date,NaN
company_address_1,********NO ADDRESS DETAILS*******
company_address_2,********NO ADDRESS DETAILS*******


Only one company has the date 1753. Its record has no address details and several other unrealistic dates, so the 1753 date is a placeholder, not a real registration date. 157 companies were registered before 1900. The next check compares older and newer registration periods, to decide which years the analysis should use.

#### Registration periods: choosing the start year
Here I group companies by 5-year registration period and check two things for each period: how many companies have the status `Dissolved-20 years`, and how many have a sector code.

In [12]:
# Registration year from the converted registration date
reg_year = reg_date.dt.year

# 5-year registration period (for example 2003 becomes 2000)
period = ((reg_year // 5) * 5).rename("period")

# Companies with the status Dissolved-20 years, and companies with any sector code
is_dissolved_20y = companies["company_status"].str.strip() == "Dissolved-20 years"
has_sector = companies["nace_v2_code"].notna() | companies["princ_object_code"].notna()

# Count, Dissolved-20 years and % with a sector code, per period
period_summary = pd.DataFrame({
    "companies": period.value_counts(),
    "dissolved_20y": is_dissolved_20y.groupby(period).sum(),
    "sector_pct": (has_sector.groupby(period).mean() * 100).round(1),
}).sort_index()

period_summary[period_summary.index >= 1960]

,companies,dissolved_20y,sector_pct
period,,,
1960.0,2406,118,10.8
1965.0,5421,238,10.6
1970.0,16624,1103,6.6
1975.0,22204,2194,6.6
1980.0,32401,731,6.8
1985.0,48625,1183,88.6
1990.0,74203,2528,99.7
1995.0,91859,5894,99.4
2000.0,76562,1090,99.6


For companies registered between 1960 and 1984, only about 7–11% have a sector code, so I cannot use them for the sector questions. The status `Dissolved-20 years` is only found in periods up to 2000–2004. It is a register label for very old closures, not a normal business closure, so it makes the older periods hard to compare with newer ones. Because of this, I use companies registered from 2000 onwards. The 2000–2004 period still has 1,090 of these companies, so in notebook 3 I will check if the results change when I start from 2005 instead.

#### Companies with no registration date
16,192 companies have no registration date. Here I check their type and status, to see what kind of companies they are.

In [13]:
# True for companies with no registration date
no_reg_date = companies["company_reg_date"].isna()

# Their 5 most common types and statuses
display(companies.loc[no_reg_date, "company_type"].value_counts().head(5))
display(companies.loc[no_reg_date, "company_status"].value_counts().head(5))

company_type
Unknown                                 14414
Private                                  1283
Private unlimited                         255
Private limited by shares                 171
Private unlimited with share capital       49
Name: count, dtype: int64

company_status
Dissolved              13899
Dissolved-20 years      2288
Ceased IRL                 2
Ceased                     1
Conv. to IndProvSoc        1
Name: count, dtype: int64

Most companies with no registration date have the type `Unknown` (14,414) or one of the old long labels, and almost all are `Dissolved` or `Dissolved-20 years`. So these are closed companies with old records. Without a registration date I cannot place them in a registration year, so they cannot be used for survival analysis and will be removed when I keep companies registered from 2000 onwards.

#### Dissolution date by status
To measure how long a company survived, I need a closing date. Here I check, for each status, how many companies have a `comp_dissolved_date`.

In [14]:
# Status without extra spaces (only for this comparison)
status = companies["company_status"].str.strip()

# True when the company has a dissolution date
has_dissolved_date = companies["comp_dissolved_date"].notna()

# Number of companies per status, with (True) and without (False) a dissolution date, largest status first
dissolved_by_status = pd.crosstab(status, has_dissolved_date)
dissolved_by_status.loc[status.value_counts().index]

comp_dissolved_date,False,True
company_status,,
Dissolved,1019,457782
Normal,324214,188
Dissolved-20 years,2084,15569
Strike Off Listed,10066,740
Liquidation,9157,54
Ceased IRL,2676,0
Dissolved PostMerger,2100,0
Ceased,1254,0
Struck Off,14,673


Almost all `Dissolved` companies have a dissolution date (457,782), and only 1,019 do not. Most `Normal` companies have no dissolution date, as expected, but 188 have one. `Struck Off` companies mostly have a date. `Dissolved PostMerger`, `Ceased`, `Ceased IRL` and the small statuses never have one. `Strike Off Listed` and `Liquidation` mostly have no date, because these companies are still in the process of closing. Before deciding which statuses count as closed, I need to check what date the statuses without a dissolution date can use.

#### Status date by status
Some closed statuses have no dissolution date. Here I check `company_status_date` (the date the current status started) for each status, and whether it matches the dissolution date when both exist.

In [15]:
# True when the company has a status date
has_status_date = companies["company_status_date"].notna()

# Number of companies per status, with (True) and without (False) a status date
status_date_by_status = pd.crosstab(status, has_status_date)
display(status_date_by_status.loc[status.value_counts().index])

# For companies with both dates: is the status date the same as the dissolution date?
both_dates = has_dissolved_date & has_status_date
same_date = (companies["comp_dissolved_date"] == companies["company_status_date"]).rename("same_date")
display(pd.crosstab(status[both_dates], same_date[both_dates]))

company_status_date,False,True
company_status,,
Dissolved,765,458036
Normal,316976,7426
Dissolved-20 years,2084,15569
Strike Off Listed,0,10806
Liquidation,0,9211
Ceased IRL,0,2676
Dissolved PostMerger,0,2100
Ceased,1100,154
Struck Off,0,687


same_date,False,True
company_status,,
Dissolved,0,457782
Dissolved-20 years,0,15569
Liquidation,54,0
Normal,61,127
Strike Off Listed,740,0
Struck Off,673,0


For `Dissolved` and `Dissolved-20 years` companies, the status date is always the same as the dissolution date. Statuses that have no dissolution date, such as `Dissolved PostMerger` and `Ceased IRL`, do have a status date. So the status date can be used as the closing date for all closed statuses. For `Struck Off`, `Strike Off Listed` and `Liquidation`, the two dates are never the same, so the dissolution date in these rows refers to an earlier event. For these statuses the status date is the date the current status started. `Ceased` companies mostly have no status date (1,100). 7,426 `Normal` companies also have a status date.

#### One row per company
Each company should appear once. Here I check for repeated company numbers and fully repeated rows.

In [16]:
# Rows that are exact copies of another row
print("Fully repeated rows:", companies.duplicated().sum())

# Company numbers that appear more than once
repeated = companies["company_num"].duplicated(keep=False)
print("Rows with a repeated company number:", repeated.sum())

# Those rows, sorted so the pairs are next to each other
columns_to_show = ["company_num", "company_name", "company_status", "company_type",
                   "company_reg_date", "comp_dissolved_date", "company_status_date"]
companies.loc[repeated, columns_to_show].sort_values("company_num")

Fully repeated rows: 0
Rows with a repeated company number: 10


,company_num,company_name,company_status,company_type,company_reg_date,comp_dissolved_date,company_status_date
131509,566202,NAIP MEMOS LIMITED,Normal,LTD - Private Company Limited by Shares,2015-08-07,NaN,2026-04-27
447648,566202,NAIP MEMOS LIMITED,Normal,LTD - Private Company Limited by Shares,2015-08-07,NaN,2024-01-10
607552,566306,MARK DOHERTY ELECTRICAL LIMITED,Normal,LTD - Private Company Limited by Shares,2015-08-10,NaN,NaN
54953,566306,MARK DOHERTY ELECTRICAL LIMITED,Normal,LTD - Private Company Limited by Shares,2015-08-10,NaN,2026-04-27
607551,753407,FEZ DRINKS LIMITED,Normal,LTD - Private Company Limited by Shares,2023-12-06,NaN,2026-05-04
46150,753407,FEZ DRINKS LIMITED,Normal,LTD - Private Company Limited by Shares,2023-12-06,2026-05-11,2026-05-04
163375,758130,JSN HOSPITALITY LIMITED,Normal,LTD - Private Company Limited by Shares,2024-02-20,NaN,NaN
591013,758130,JSN HOSPITALITY LIMITED,Normal,LTD - Private Company Limited by Shares,2024-02-20,NaN,NaN
613287,800043,BRISTOL-MYERS SQUIBB / JOHNSON & JOHNSON EEIG,Normal,European economic interest grouping,2026-10-12,NaN,2026-10-12
685379,800043,BKK ACCOUNT SERVICES LIMITED,Normal,LTD - Private Company Limited by Shares,2025-10-28,NaN,2026-10-12


There are no full duplicate rows. 5 company numbers appear twice. For 566202, 566306, 753407 and 758130, both rows are the same company. Number 800043 is used by two different companies: a European economic interest grouping and BKK ACCOUNT SERVICES LIMITED. When cleaning, I keep one row for each of the four repeated companies. For number 800043 I keep the limited company and remove the European economic interest grouping, which was registered in October 2026 and is too recent for the survival analysis. After this, the company number is unique.

#### Sector codes by registration year
There are two sector code columns: `princ_object_code` (older NACE Rev 1.1 codes) and `nace_v2_code` (newer NACE Rev 2 codes). Here I check what percentage of companies have each code, for each registration year from 2000.

In [17]:
# Registration year
reg_year = reg_date.dt.year

# % of companies with each sector code, per registration year
sector_by_year = pd.DataFrame({
    "princ_object_pct": companies["princ_object_code"].notna().groupby(reg_year).mean() * 100,
    "nace_v2_pct": companies["nace_v2_code"].notna().groupby(reg_year).mean() * 100,
}).round(1)

sector_by_year[sector_by_year.index >= 2000]

,princ_object_pct,nace_v2_pct
company_reg_date,,
2000.0,99.4,0.0
2001.0,99.6,0.1
2002.0,99.6,0.1
2003.0,99.6,0.0
2004.0,99.5,0.1
2005.0,99.6,0.1
2006.0,99.6,0.1
2007.0,99.7,0.1
2008.0,99.6,0.1


The sector code system changes in 2015. Companies registered up to 2014 have the old code (`princ_object_code`, about 99%) and almost never the new one. From 2016 almost all have the new code (`nace_v2_code`, about 99%). In 2015, companies are split between the two (38.5% old, 60.8% new). Because the two systems use different codes, I cannot compare sectors across the change without converting one into the other. So the sector analysis uses companies registered from 2016 onwards, with `nace_v2_code` only.

#### Format of the NACE Rev 2 codes
NACE Rev 2 codes have 4 digits (for example 5630 = beverage serving activities). `nace_v2_code` is stored as a number, so I check the range of values and whether any codes have lost a leading zero.

In [18]:
# Filled codes only
nace = companies["nace_v2_code"].dropna()

print("Different codes:", nace.nunique())
print("Smallest:", nace.min(), " Largest:", nace.max())
print("Codes equal to 0:", (nace == 0).sum())
print("Codes below 1000 (3 digits or fewer):", (nace < 1000).sum())

# The most common codes below 1000
nace[nace < 1000].value_counts().head(3)

Different codes: 582
Smallest: 0.0  Largest: 9900.0
Codes equal to 0: 10
Codes below 1000 (3 digits or fewer): 8263


nace_v2_code
150.0    2959
141.0    2050
143.0     745
Name: count, dtype: int64

`nace_v2_code` has 582 different codes, from 0 to 9900. 8,263 codes are below 1000 (this includes the 10 zeros), for example 150 and 141. NACE codes have 4 digits, so these codes lost their leading zero when the CSV was saved (150 should be 0150). A code of 0 is not a real sector. When cleaning, I store the code as 4-digit text with the leading zero and set 0 to missing.

#### Address: where the county is
There is no county column, so the county has to come from the address. Here I check the last address line, `company_address_4`.

In [19]:
# Number of different values in the last address line
print("Different values:", companies["company_address_4"].nunique())

# The 15 most common values
companies["company_address_4"].value_counts().head(15)

Different values: 15949


company_address_4
DUBLIN                       76003
DUBLIN, Ireland              40571
DUBLIN 2, DUBLIN             21536
DUBLIN 2, DUBLIN, IRELAND    18118
CORK                         13889
DUBLIN, IRELAND              12385
DUBLIN 2                     10054
CO. CORK,CORK                 7053
Dublin 2                      6948
CO. CORK, CORK, IRELAND       6732
, IRELAND                     6213
GALWAY                        6143
CO. DUBLIN,DUBLIN             5092
,DUBLIN                       4631
DUNDALK,LOUTH                 4213
Name: count, dtype: int64

The last address line has 15,949 different values, because the same place is written in many ways: upper and lower case (`DUBLIN 2`, `Dublin 2`), with or without `CO.` and `IRELAND`, and with extra commas. Most values still contain a county name, for example `DUBLIN`, `CORK`, `GALWAY` or `LOUTH`. Some, like `, IRELAND`, have no county. So I can find the county by searching each address for one of the 26 county names, and use `company_address_3` when line 4 has no county.

#### Can the county be found from the address?
I search each address for one of the 26 county names. Town names can contain a county name (for example `CLAREMORRIS, MAYO`), so I split the address at the commas and check the parts from the end, where the county usually is. `WESTMEATH` is checked before `MEATH`, because `MEATH` is part of `WESTMEATH`.

In [20]:
# The 26 counties of Ireland (WESTMEATH before MEATH)
counties = ["CARLOW", "CAVAN", "CLARE", "CORK", "DONEGAL", "DUBLIN", "GALWAY", "KERRY",
            "KILDARE", "KILKENNY", "LAOIS", "LEITRIM", "LIMERICK", "LONGFORD", "LOUTH",
            "MAYO", "WESTMEATH", "MEATH", "MONAGHAN", "OFFALY", "ROSCOMMON", "SLIGO",
            "TIPPERARY", "WATERFORD", "WEXFORD", "WICKLOW"]


def find_county(address):
    """Return the county found in an address line, or None if there is none."""
    if pd.isna(address):
        return None
    parts = address.upper().split(",")
    for part in reversed(parts):
        for county in counties:
            if county in part:
                return county
    return None


# County from line 4, and from line 3 when line 4 has none
county_line_4 = companies["company_address_4"].apply(find_county)
county_line_3 = companies["company_address_3"].apply(find_county)
county = county_line_4.fillna(county_line_3)

# How many companies registered from 2000 get a county
from_2000 = reg_date >= "2000-01-01"
print("Companies registered from 2000:", from_2000.sum())
print("County found in line 4:", county_line_4[from_2000].notna().sum())
print("County found only in line 3:", (county_line_4[from_2000].isna() & county_line_3[from_2000].notna()).sum())
print("No county found:", county[from_2000].isna().sum())

Companies registered from 2000: 512042
County found in line 4: 504038
County found only in line 3: 6182
No county found: 1822


Of the 512,042 companies registered from 2000, the county is found in line 4 for 504,038 and only in line 3 for another 6,182. For 1,822 companies no county is found. Checking the parts from the end gives the right county for addresses like `CLAREMORRIS, MAYO` (Mayo, not Clare) and `ENNISKERRY, WICKLOW` (Wicklow, not Kerry).

#### Checking the county against the Eircode (Dublin only)
Dublin postal districts have their own Eircode routing keys (`D01`–`D24` and `D6W`), which are all in County Dublin. For companies with one of these Eircodes, I check whether the address also gives Dublin.

In [21]:
# Eircode in upper case without spaces; the routing key is the first 3 characters
routing_key = companies["eircode"].str.upper().str.replace(" ", "").str[:3]

# Dublin routing keys: D01 to D24 and D6W
dublin_keys = [f"D{n:02d}" for n in range(1, 25)] + ["D6W"]
has_dublin_eircode = from_2000 & routing_key.isin(dublin_keys)

print("Companies from 2000 with a Dublin Eircode:", has_dublin_eircode.sum())
county[has_dublin_eircode].value_counts(dropna=False).head(6)

Companies from 2000 with a Dublin Eircode: 119164


company_address_4
DUBLIN     117363
NaN           666
MEATH         332
CORK          137
WICKLOW        83
KILDARE        77
Name: count, dtype: int64

Of the 119,164 companies with a Dublin Eircode, the address also gives Dublin for 117,363. Only a small number get another county or no county from the address. This supports using the address to find the county, especially for the Dublin vs other counties comparison. Some Dublin areas have Eircodes that do not start with D (for example A94 Blackrock), so this check covers Dublin postal districts only, not the whole county.

#### Summary of findings

The file has one row per company, except for 5 repeated company numbers. The registration date is filled and valid for all but 16,192 companies, which are old closed companies. Before 1985 most companies have no sector code, and the status `Dissolved-20 years` affects periods up to 2000–2004, so I use companies registered from 2000. The sector code system changes in 2015, so sector analysis uses 2016 onwards with `nace_v2_code`, which lost its leading zeros. For companies registered from 2000, every status other than `Normal` has a status date, and this date matches the dissolution date where both refer to the same event. The county can be found from the address for almost all companies from 2000.

**Cleaning plan**
1. Remove extra spaces from the text columns.
2. Convert the registration date and status date to dates.
3. Keep companies registered from 2000 onwards (this also removes the missing and placeholder dates).
4. Keep one row per company number (the 4 repeated companies and the 800043 case).
5. Mark a company as closed if it has left the register, with `company_status_date` as the closing date. Companies that are `Normal`, being struck off (`Strike Off Listed`), in liquidation, in examinership or in administration are still on the register, so they count as not closed. Companies that left the register through a merger or a move abroad count as closed.
6. Group the 44 company types into a few main types, putting old and new labels of the same type together.
7. Store `nace_v2_code` as 4-digit text with the leading zero added back, and treat 0 as missing.
8. Add the county from the address, using the `find_county` function.
9. Keep only the columns needed and give them short names.

## 2. Company Records: cleaning

I clean a copy of the raw data, so `companies` stays as it was loaded. I follow the cleaning plan above, one item at a time, and check the row count after each step.

#### Remove spaces in text columns

Some text values have extra spaces, for example the status `Normal ` with a trailing space. I remove spaces at the start and end of every text column.

In [22]:
# Copy of the raw data for cleaning
companies_clean = companies.copy()

# Text columns
text_cols = companies_clean.select_dtypes(include="str").columns
print(len(text_cols), "text columns")

# Remove spaces at the start and end of each text value
for col in text_cols:
    companies_clean[col] = companies_clean[col].str.strip()

# Check: number of status values and the Normal count
print(companies_clean["company_status"].nunique(), "status values")
companies_clean["company_status"].value_counts().head(3)

17 text columns
22 status values


company_status
Dissolved             458801
Normal                324402
Dissolved-20 years     17653
Name: count, dtype: int64

After removing spaces, `company_status` has 22 values instead of 23. The two Normal labels are now one, with 324,402 companies.

#### Convert dates

`company_reg_date` and `company_status_date` are stored as text. I convert them to dates, so I can filter by year and measure how long a company stayed on the register. In the inspection, no registration date failed to convert, so the number of missing values should not change.

In [23]:
# Check from the last step: no status ends with a space
assert not companies_clean["company_status"].str.endswith(" ").any()

# Convert the two date columns and compare missing values before and after
date_cols = ["company_reg_date", "company_status_date"]

for col in date_cols:
    missing_before = companies_clean[col].isna().sum()
    companies_clean[col] = pd.to_datetime(companies_clean[col], errors="coerce")
    missing_after = companies_clean[col].isna().sum()
    print(col, "missing before:", missing_before, "after:", missing_after)

# First and last registration date
print(companies_clean["company_reg_date"].min(), companies_clean["company_reg_date"].max())

company_reg_date missing before: 16192 after: 16192
company_status_date missing before: 320925 after: 320925
1753-01-01 00:00:00 2026-10-27 00:00:00


Both date columns are now dates. The number of missing values did not change, so every date converted. Registration dates go from 1753-01-01 (the placeholder record) to 2026-10-27.

#### Keep companies registered from 2000

The inspection showed that sector codes and the status `Dissolved-20 years` are not reliable for older companies, so I start the analysis in 2000. This also removes the companies with no registration date and the placeholder record from 1753.

In [24]:
# Keep companies registered from 2000 onwards
rows_before = len(companies_clean)
companies_clean = companies_clean[companies_clean["company_reg_date"].dt.year >= 2000]

print("Rows before:", rows_before, "after:", len(companies_clean))
print("Missing registration dates:", companies_clean["company_reg_date"].isna().sum())
print("First registration date:", companies_clean["company_reg_date"].min())

Rows before: 827902 after: 512042
Missing registration dates: 0
First registration date: 2000-01-01 00:00:00


After the filter, 512,042 companies are left, all registered from 2000 onwards. No registration dates are missing.

#### One row per company number

Five company numbers appear twice. For number 800043, I remove the European economic interest grouping and keep the limited company. For the other four, both rows are the same company, so I keep the row with the latest status date.

In [25]:
# Rows that share a company number
repeated = companies_clean["company_num"].duplicated(keep=False)
print("Rows with a repeated company number:", repeated.sum())

# Number 800043: remove the European economic interest grouping
is_eeig = (companies_clean["company_num"] == 800043) & ~companies_clean["company_name"].str.contains("LIMITED")
companies_clean = companies_clean[~is_eeig]

# Other repeats: keep the row with the latest status date
companies_clean = companies_clean.sort_values("company_status_date", na_position="first")
companies_clean = companies_clean.drop_duplicates(subset="company_num", keep="last")

print("Rows after:", len(companies_clean))
print("Company number is unique:", companies_clean["company_num"].is_unique)

Rows with a repeated company number: 10
Rows after: 512037
Company number is unique: True


10 rows shared a company number. After removing the European economic interest grouping and keeping one row for each of the other four companies, 512,037 companies are left and each company number appears once.

#### Closed companies and closing date

A company counts as closed when it has left the register. Companies that are Normal, Strike Off Listed, in liquidation, in examinership or in administration are still on the register, so they are not closed. All other statuses, including mergers and moves abroad, mean the company has left the register. The closing date is `company_status_date`.

In [26]:
# Statuses where the company is still on the register
still_registered = ["Normal", "Strike Off Listed", "Liquidation", "Liquidation (UK)",
                    "Liquidation stayed", "Examinership", "Administration", "Administration (UK)"]

# Check: names in the list that do not match any status in the data
print("Not found in data:", set(still_registered) - set(companies_clean["company_status"]))

# Closed flag and closing date (missing for companies still registered)
companies_clean["closed"] = ~companies_clean["company_status"].isin(still_registered)
companies_clean["closed_date"] = companies_clean["company_status_date"].where(companies_clean["closed"])

# Each status and whether it counts as closed
display(pd.crosstab(companies_clean["company_status"], companies_clean["closed"]))

# Check: closed companies without a closing date
print("Closed without a date:", (companies_clean["closed"] & companies_clean["closed_date"].isna()).sum())

# Number of companies still registered (False) and closed (True)
companies_clean["closed"].value_counts()

Not found in data: {'Liquidation (UK)', 'Liquidation stayed'}


closed,False,True
company_status,,
Administration,1,0
Administration (UK),7,0
Ceased,0,72
Ceased IRL,0,1243
Conv. to IndProvSoc,0,8
De-Registered ICAV,0,65
De-Registered Investment Company,0,1
Deleted CB Conversion,0,7
Deleted CB Merger,0,106


Closed without a date: 0


closed
False    305953
True     206084
Name: count, dtype: int64

Every status is now either still registered or closed. `Liquidation stayed` and `Liquidation (UK)` are in the list but do not appear in companies registered from 2000. 305,953 companies are still on the register and 206,084 have closed. All closed companies have a closing date.

#### Main company type

`company_type` has 44 labels. Some are old names and some are new short codes for the same kind of company, for example a private company limited by shares and LTD. I put them into a few main types by looking for key words in each label.

In [27]:
# Main type from key words in the label (checked in this order)
def find_type(label):
    label = str(label).lower()
    if "unknown" in label:
        return "Unknown"
    if "variable capital" in label:
        return "Other"
    if "designated activity" in label or "dac" in label:
        return "DAC"
    if "guarantee" in label or "g/tee" in label or "clg" in label:
        return "Guarantee (CLG)"
    if "unlimited" in label or "ulc" in label:
        return "Unlimited"
    if "public" in label or "plc" in label:
        return "Public (PLC)"
    if "private" in label or "ltd" in label:
        return "Private limited (LTD)"
    return "Other"

companies_clean["type_group"] = companies_clean["company_type"].apply(find_type)

# Each label with its main type and number of companies
display(companies_clean[["type_group", "company_type"]].value_counts().sort_index())

# Number of companies per main type
companies_clean["type_group"].value_counts()

type_group             company_type                                                               
DAC                    DAC - Designated Activity Company (limited by guarantee)                          106
                       DAC - Designated Activity Company (limited by shares)                           10653
                       DAC- Designated Activity Company (limited by guarantee ) (licenced company)         5
                       DAC-Designated Activity Company (limited by shares) (Licenced Company)             23
Guarantee (CLG)        CLG - Company Limited by Guarantee                                              17912
                       CLG-Company Limited by Guarantee (licenced company)                              1240
                       Guarantee                                                                           7
                       Guarantee company without a share capital (public)                               3893
                       Guaran

type_group
Private limited (LTD)    464700
Guarantee (CLG)           23224
DAC                       10787
Unlimited                  6590
Other                      5499
Public (PLC)               1235
Unknown                       2
Name: count, dtype: int64

Each of the 44 labels now belongs to one main type. Most companies are private companies limited by shares (464,700). Public limited companies with variable capital are investment funds, so I put them in Other with the other fund types.

#### NACE code as 4-digit text

`nace_v2_code` is stored as a number, so codes such as 0150 lost their leading zero and show as 150. I store the code as 4-digit text with the leading zero. Code 0 is not a real sector, so I set it to missing.

In [28]:
# NACE code: 0 becomes missing
nace = companies_clean["nace_v2_code"]
nace = nace.where(nace != 0)

# Whole number, then text with leading zeros to 4 digits (missing stays missing)
companies_clean["nace_code"] = nace.dropna().astype(int).astype(str).str.zfill(4)

# Checks
print("Missing before:", companies_clean["nace_v2_code"].isna().sum())
print("Zeros set to missing:", (companies_clean["nace_v2_code"] == 0).sum())
print("Missing after:", companies_clean["nace_code"].isna().sum())
print("Code length:", companies_clean["nace_code"].str.len().value_counts().to_dict())
print("Codes starting with 0:", companies_clean["nace_code"].str.startswith("0").sum())
companies_clean["nace_code"].dropna().head(3)

Missing before: 246138
Zeros set to missing: 5
Missing after: 246143
Code length: {4.0: 265894}
Codes starting with 0: 8251


39    7735
42    6499
43    6832
Name: nace_code, dtype: str

`nace_code` now holds the sector code as 4-digit text. Code 0 is missing, and codes that lost their leading zero have it back (150 is now 0150).

#### County from the address

I find the county with the `find_county()` function from the inspection. I check address line 4 first and use line 3 when line 4 has no county. Companies with no county in either line keep a missing county.

In [29]:
# County from address line 4, then from line 3 where line 4 has none
county_line4 = companies_clean["company_address_4"].apply(find_county)
county_line3 = companies_clean["company_address_3"].apply(find_county)
companies_clean["county"] = county_line4.fillna(county_line3)

# Checks
print("Found in line 4:", county_line4.notna().sum())
print("Found in line 3 only:", (county_line4.isna() & county_line3.notna()).sum())
print("No county:", companies_clean["county"].isna().sum())
print("Different counties:", companies_clean["county"].nunique(), "of", len(counties))

Found in line 4: 504033
Found in line 3 only: 6182
No county: 1822
Different counties: 26 of 26


I found a county for most companies: 504,033 from address line 4 and 6,182 more from line 3. 1,822 companies have no county in either line and keep a missing county.

#### Keep the needed columns

I keep only the columns the analysis needs and give them short names. The raw sector code, the old 44 type labels and the address lines are not needed after the steps above.

In [30]:
# Columns to keep: old name -> new name
keep = {
    "company_num": "company_num",
    "company_reg_date": "reg_date",
    "type_group": "company_type",
    "company_status": "status",
    "closed": "closed",
    "closed_date": "closed_date",
    "nace_code": "nace_code",
    "county": "county",
}

companies_clean = companies_clean[list(keep)].rename(columns=keep)
companies_clean = companies_clean.sort_values("company_num").reset_index(drop=True)

companies_clean.info()
companies_clean.head(3)

<class 'pandas.DataFrame'>
RangeIndex: 512037 entries, 0 to 512036
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   company_num   512037 non-null  int64         
 1   reg_date      512037 non-null  datetime64[us]
 2   company_type  512037 non-null  str           
 3   status        512037 non-null  str           
 4   closed        512037 non-null  bool          
 5   closed_date   206084 non-null  datetime64[us]
 6   nace_code     265894 non-null  str           
 7   county        510215 non-null  str           
dtypes: bool(1), datetime64[us](2), int64(1), str(4)
memory usage: 27.8 MB


,company_num,reg_date,company_type,status,closed,closed_date,nace_code,county
0,500,2000-01-01,Other,Normal,False,NaT,NaN,NaN
1,17805,2021-01-26,Private limited (LTD),Dissolved PostMerger,True,1959-10-22,NaN,CAVAN
2,22031,2021-01-26,Private limited (LTD),Dissolved PostMerger,True,2005-06-01,NaN,CAVAN


The cleaned table has 512,037 companies and 8 columns. `closed_date` is filled for the 206,084 closed companies only.

#### Check: closing date after registration

A company cannot close before it is registered. I check that no closing date is earlier than the registration date.

In [31]:
# Closed companies with a closing date before the registration date
closed_before_reg = companies_clean["closed_date"] < companies_clean["reg_date"]

print("Closed before registration:", closed_before_reg.sum())
companies_clean[closed_before_reg].head(3)

Closed before registration: 29


,company_num,reg_date,company_type,status,closed,closed_date,nace_code,county
1,17805,2021-01-26,Private limited (LTD),Dissolved PostMerger,True,1959-10-22,NaN,CAVAN
2,22031,2021-01-26,Private limited (LTD),Dissolved PostMerger,True,2005-06-01,NaN,CAVAN
3,77224,2021-05-10,Unlimited,Dissolved PostMerger,True,2020-01-17,NaN,MONAGHAN


In [32]:
# The 29 problem rows: company numbers, statuses and registration years
problem = companies_clean[closed_before_reg]

print(problem["company_num"].describe())
print(problem["status"].value_counts())
print(problem["reg_date"].dt.year.value_counts())

count        29.000000
mean     352356.793103
std      276225.644169
min       17805.000000
25%      135867.000000
50%      238719.000000
75%      595526.000000
max      908608.000000
Name: company_num, dtype: float64
status
Dissolved PostMerger    26
Ceased IRL               3
Name: count, dtype: int64
reg_date
2021    23
2020     5
2001     1
Name: count, dtype: int64


#### Remove companies that closed before registration

29 companies have a closing date before their registration date. All of them left the register after a merger or a move abroad (26 Dissolved PostMerger, 3 Ceased IRL), and 28 have a registration date in 2020 or 2021. This suggests their registration date was changed when the merger was recorded. Their lifetime cannot be measured, so I remove them.

In [33]:
# Remove the companies with a closing date before the registration date
rows_before = len(companies_clean)
companies_clean = companies_clean[~closed_before_reg].reset_index(drop=True)

print("Rows before:", rows_before, "after:", len(companies_clean))
print("Closed companies:", companies_clean["closed"].sum())
assert (companies_clean["closed_date"] < companies_clean["reg_date"]).sum() == 0

Rows before: 512037 after: 512008
Closed companies: 206055


#### Cleaning summary

1. **I removed extra spaces in all text columns.** `company_status` now has 22 values instead of 23.
2. **I converted the registration date and the status date to dates.** No dates failed to convert.
3. **I kept companies registered from 2000 onwards.** This left 512,042 of the 827,902 rows and removed all rows without a registration date.
4. **I kept one row per company number.** I removed the European economic interest grouping under number 800043 and one row for each of the other four repeated companies. 512,037 companies were left.
5. **I marked each company as closed or still registered.** A company is closed when it has left the register, and its closing date is `company_status_date`. All closed companies have a closing date.
6. **I grouped the 44 company type labels into 7 main types.** Investment funds with variable capital are in Other.
7. **I stored the sector code as 4-digit text with the leading zero.** Code 0 is now missing.
8. **I found the county from address line 4, or line 3 when line 4 has none.** 1,822 companies have no county.
9. **I kept 8 columns with short names.**
10. **I removed 29 companies that closed before their registration date.** All were mergers or moves abroad.

The cleaned table has 512,008 companies, of which 206,055 have closed.

## 3. Save the cleaned data

I save the cleaned table to `data/processed/companies_clean.csv`. Then I load it again to check that the rows, the dates and the sector codes with leading zeros came through correctly.

In [34]:
# Save the cleaned table
companies_clean.to_csv("../data/processed/companies_clean.csv", index=False)

# Load it again and check
check = pd.read_csv("../data/processed/companies_clean.csv",
                    dtype={"nace_code": "str"},
                    parse_dates=["reg_date", "closed_date"])

print("Shape:", check.shape)
print("Closed:", check["closed"].sum())
print("Code length:", check["nace_code"].str.len().value_counts().to_dict())
print("Codes starting with 0:", check["nace_code"].str.startswith("0").sum())
check.dtypes

Shape: (512008, 8)
Closed: 206055
Code length: {4.0: 265889}
Codes starting with 0: 8251


company_num              int64
reg_date        datetime64[us]
company_type               str
status                     str
closed                    bool
closed_date     datetime64[us]
nace_code                  str
county                     str
dtype: object

The reloaded file has 512,008 rows and 8 columns. The dates are read as dates and every sector code still has 4 digits with its leading zero.